In [27]:
import time
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50, DenseNet121, MobileNetV3Large
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import average_precision_score
from google.colab import drive

#Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#Paths
BASE_PATH = '/content/drive/MyDrive/animal_dataset gp18/'
TRAIN_DIR = BASE_PATH + 'dataset/train'
VAL_DIR = BASE_PATH + 'dataset/val'
TEST_DIR = BASE_PATH + 'dataset/test'

#Image settings
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_EPOCHS = 50
FINE_TUNE_EPOCHS = 10

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Train path: {TRAIN_DIR}")
print(f"Val path: {VAL_DIR}")
print(f"Test path: {TEST_DIR}")
print(f"Image size: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Fine-tune epochs: {FINE_TUNE_EPOCHS}")

#Check GPU
print(f"\nGPU Available: {tf.config.list_physical_devices('GPU')}")
print(f"TensorFlow version: {tf.__version__}")

CONFIGURATION
Train path: /content/drive/MyDrive/animal_dataset gp18/dataset/train
Val path: /content/drive/MyDrive/animal_dataset gp18/dataset/val
Test path: /content/drive/MyDrive/animal_dataset gp18/dataset/test
Image size: (224, 224)
Batch size: 32
Epochs: 50
Fine-tune epochs: 10

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow version: 2.20.0


In [10]:
from pathlib import Path

print("\n" + "="*60)
print("LOADING DATASETS")
print("="*60)

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

NUM_CLASSES = len(train_dataset.class_names)

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

#Calculate class weights to handle imbalanced data
print("\n--- Calculating class weights for imbalance correction ---")

# Get all labels from training dataset
y_train = []
for images, labels in train_dataset:
    # Convert one-hot labels to class indices
    y_train.extend(np.argmax(labels.numpy(), axis=1))

# Calculate balanced class weights
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

print(f"✓ Class weights calculated for {len(class_weight_dict)} classes")
print(f"  Sample weights - Class 0: {class_weight_dict[0]:.3f}, Class 1: {class_weight_dict[1]:.3f}...")

print(f"\n✓ Number of classes: {NUM_CLASSES}")

#Show class distribution
train_counts = {}
for class_name in train_dataset.class_names:
    count = len([f for f in Path(TRAIN_DIR + '/' + class_name).glob('*') if f.is_file()])
    train_counts[class_name] = count

print(f"\nClass distribution (training set):")
for class_name, count in sorted(train_counts.items(), key=lambda x: x[1]):
    print(f"  {class_name}: {count} images")

#Optimize dataset performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(AUTOTUNE)
val_dataset = val_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)


LOADING DATASETS
Found 404 files belonging to 18 classes.
Found 80 files belonging to 18 classes.
Found 105 files belonging to 18 classes.

--- Calculating class weights for imbalance correction ---
✓ Class weights calculated for 17 classes
  Sample weights - Class 0: 3.395, Class 1: 11.882...

✓ Number of classes: 18

Class distribution (training set):
  tiger_indochinese: 0 images
  wolf_grey: 1 images
  eagle_golden: 2 images
  lion_asiatic: 2 images
  lion_african: 4 images
  tiger_sumatran: 4 images
  eagle_bald: 7 images
  eagle_harpy: 7 images
  wolf_arctic: 11 images
  eagle_steppe: 12 images
  leopard_clouded: 15 images
  tiger_siberian: 15 images
  leopard_snow: 23 images
  lion_white: 23 images
  leopard_african: 45 images
  tiger_bengal: 48 images
  wolf_ethiopian: 56 images
  leopard_amur: 129 images


In [12]:
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.1),
])

print("\n✓ Data augmentation configured")


✓ Data augmentation configured


In [17]:
def build_model(model_name):

    print(f"\n--- Building {model_name} ---")

    #Select pre-trained base model
    if model_name == 'ResNet50':
        base_model = ResNet50(
            weights='imagenet',
            include_top=False,
            input_shape=(224, 224, 3)
        )
        print("  Base model: ResNet50 (23.5M parameters)")

    elif model_name == 'DenseNet121':
        base_model = DenseNet121(
            weights='imagenet',
            include_top=False,
            input_shape=(224, 224, 3)
        )
        print("  Base model: DenseNet121 (7.0M parameters)")

    elif model_name == 'MobileNetV3':
        base_model = MobileNetV3Large(
            weights='imagenet',
            include_top=False,
            input_shape=(224, 224, 3)
        )
        print("  Base model: MobileNetV3 (5.4M parameters)")

    else:
        raise ValueError(f"Unknown model: {model_name}")

    #Freeze base model weights (for initial training)
    base_model.trainable = False

    #Build complete model
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = data_augmentation(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)

    # Compile
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall')
        ]
    )

    return model, base_model

In [18]:
def fine_tune_model(model, base_model, train_dataset, val_dataset):
    print(f"\n--- Fine-tuning ({FINE_TUNE_EPOCHS} epochs) ---")

    # Unfreeze the base model
    base_model.trainable = True

    # Recompile with lower learning rate
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=FINE_TUNE_EPOCHS,
        verbose=1
    )

    return history

In [19]:
def calculate_map(model, dataset):
    print("  Calculating mAP (this may take a few minutes)...")
    all_preds = []
    all_labels = []

    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    ap_scores = []
    for i in range(NUM_CLASSES):
        ap = average_precision_score(all_labels[:, i], all_preds[:, i])
        ap_scores.append(ap)

    mAP = np.mean(ap_scores)
    return mAP, ap_scores


In [25]:
from tensorflow.keras.applications import ResNet50, DenseNet121, MobileNetV3Large
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import time
import numpy as np
from sklearn.metrics import average_precision_score

model_names = ['ResNet50', 'DenseNet121', 'MobileNetV3']
training_records = {}
training_times = {}
fine_tune_records = {}

for model_name in model_names:
    print("\n" + "="*60)
    print(f"TRAINING {model_name}")
    print("="*60)

    # Build model
    model, base_model = build_model(model_name)

    # Callbacks
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7)
    ]

    # Phase 1: Train with frozen base (50 epochs)
    print(f"\n--- Phase 1: Training {model_name} for {NUM_EPOCHS} epochs ---")

    start_time = time.time()

    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=NUM_EPOCHS,
        callbacks=callbacks,
        verbose=1
    )

    end_time = time.time()
    training_times[model_name] = (end_time - start_time) / 60

    # Calculate mAP on validation set
    print(f"\n--- Calculating mAP for {model_name} ---")
    mAP_val, _ = calculate_map(model, val_dataset)

    # Store training history
    training_records[model_name] = {
        'history': history.history,
        'best_epoch': np.argmax(history.history['val_accuracy']) + 1,
        'best_val_accuracy': max(history.history['val_accuracy']),
        'best_val_loss': min(history.history['val_loss']),
        'mAP': mAP_val,
        'training_time_minutes': training_times[model_name]
    }

    # Save the model
    model.save(f'{model_name}_subspecies_model.keras')
    print(f"✓ {model_name} saved as '{model_name}_subspecies_model.keras'")

    # Phase 2: Fine-tuning (10 epochs)
    print(f"\n--- Phase 2: Fine-tuning {model_name} ---")
    ft_history = fine_tune_model(model, base_model, train_dataset, val_dataset)
    fine_tune_records[model_name] = ft_history

    # Save fine-tuned model
    model.save(f'{model_name}_subspecies_model_finetuned.keras')
    print(f"✓ Fine-tuned {model_name} saved as '{model_name}_subspecies_model_finetuned.keras'")


TRAINING ResNet50

--- Building ResNet50 ---
  Base model: ResNet50 (23.5M parameters)

--- Phase 1: Training ResNet50 for 50 epochs ---
Epoch 1/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 14s 512ms/step - accuracy: 0.3144 - loss: 2.5508 - precision: 0.4046 - recall: 0.1733 - val_accuracy: 0.4500 - val_loss: 1.6030 - val_precision: 0.7105 - val_recall: 0.3375 - learning_rate: 0.0010
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 269ms/step - accuracy: 0.5297 - loss: 1.6577 - precision: 0.6919 - recall: 0.3614 - val_accuracy: 0.6000 - val_loss: 1.3348 - val_precision: 0.6842 - val_recall: 0.3250 - learning_rate: 0.0010
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 263ms/step - accuracy: 0.5916 - loss: 1.4055 - precision: 0.7476 - recall: 0.3812 - val_accuracy: 0.5875 - val_loss: 1.2632 - val_precision: 0.8537 - val_recall: 0.4375 - learning_rate: 0.0010
Epoch 4/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 266ms/step - accuracy: 0.6386 - loss: 1.2257 - precision: 0.7868 - recall: 0.5025 - val_accuracy: 0.6000 - val_loss: 1

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


✓ ResNet50 saved as 'ResNet50_subspecies_model.keras'

--- Phase 2: Fine-tuning ResNet50 ---

--- Fine-tuning (10 epochs) ---
Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 56s 1s/step - accuracy: 0.6064 - loss: 1.3462 - val_accuracy: 0.6500 - val_loss: 1.1637
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step - accuracy: 0.6386 - loss: 1.2281 - val_accuracy: 0.6500 - val_loss: 1.1435
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 930ms/step - accuracy: 0.6832 - loss: 1.0905 - val_accuracy: 0.6000 - val_loss: 1.1259
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 918ms/step - accuracy: 0.7005 - loss: 1.0515 - val_accuracy: 0.6250 - val_loss: 1.1153
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 20s 894ms/step - accuracy: 0.7203 - loss: 0.9843 - val_accuracy: 0.6375 - val_loss: 1.1024
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 883ms/step - accuracy: 0.7401 - loss: 0.8932 - val_accuracy: 0.6500 - val_loss: 1.0865
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 11s 829ms/step - accuracy: 0.7723 - loss: 0.8612 - val_accuracy: 0.6

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


✓ DenseNet121 saved as 'DenseNet121_subspecies_model.keras'

--- Phase 2: Fine-tuning DenseNet121 ---

--- Fine-tuning (10 epochs) ---
Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 96s 2s/step - accuracy: 0.2673 - loss: 2.4986 - val_accuracy: 0.3875 - val_loss: 2.2642
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.3020 - loss: 2.3990 - val_accuracy: 0.3750 - val_loss: 2.2269
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.3119 - loss: 2.3324 - val_accuracy: 0.3500 - val_loss: 2.1922
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.3243 - loss: 2.2706 - val_accuracy: 0.3500 - val_loss: 2.1579
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.3292 - loss: 2.2398 - val_accuracy: 0.3125 - val_loss: 2.1198
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 20s 1s/step - accuracy: 0.3589 - loss: 2.1672 - val_accuracy: 0.3250 - val_loss: 2.0814
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.3540 - loss: 2.1369 - val_accuracy: 0.3250 - 

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


✓ MobileNetV3 saved as 'MobileNetV3_subspecies_model.keras'

--- Phase 2: Fine-tuning MobileNetV3 ---

--- Fine-tuning (10 epochs) ---
Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 40s 700ms/step - accuracy: 0.7871 - loss: 0.7692 - val_accuracy: 0.7625 - val_loss: 0.9349
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 387ms/step - accuracy: 0.7797 - loss: 0.6567 - val_accuracy: 0.7625 - val_loss: 0.9352
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 380ms/step - accuracy: 0.8094 - loss: 0.6581 - val_accuracy: 0.7750 - val_loss: 0.9340
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 338ms/step - accuracy: 0.8366 - loss: 0.5825 - val_accuracy: 0.7750 - val_loss: 0.9326
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 311ms/step - accuracy: 0.8069 - loss: 0.6469 - val_accuracy: 0.7750 - val_loss: 0.9307
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 327ms/step - accuracy: 0.7921 - loss: 0.7237 - val_accuracy: 0.7750 - val_loss: 0.9301
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 5s 382ms/step - accuracy: 0.8119 - loss: 0.5810 - val_accu

In [28]:
import pickle
import pandas as pd

with open('training_records.pkl', 'wb') as f:
    pickle.dump({
        'training_records': training_records,
        'training_times': training_times,
        'fine_tune_records': fine_tune_records
    }, f)

print("\n" + "="*60)
print("TRAINING COMPLETE - SUMMARY")
print("="*60)

summary_data = []
for model_name in model_names:
    record = training_records[model_name]
    summary_data.append({
        'Model': model_name,
        'Best Val Accuracy': f"{record['best_val_accuracy']:.4f}",
        'Best Val Loss': f"{record['best_val_loss']:.4f}",
        'mAP (val)': f"{record['mAP']:.4f}",
        'Training Time (min)': f"{record['training_time_minutes']:.2f}",
        'Best Epoch': record['best_epoch']
    })

summary_df = pd.DataFrame(summary_data)
print("\n", summary_df.to_string(index=False))


TRAINING COMPLETE - SUMMARY

       Model Best Val Accuracy Best Val Loss mAP (val) Training Time (min)  Best Epoch
   ResNet50            0.7000        1.2008    0.3516                1.08           9
DenseNet121            0.4625        1.8251    0.1756                2.78          19
MobileNetV3            0.8000        0.9368    0.4310                1.14          23


In [29]:
print("\n" + "="*60)
print("FINAL EVALUATION ON TEST SET")
print("="*60)

test_results = {}

for model_name in model_names:
    print(f"\n--- Evaluating {model_name} on test set ---")

    # Try to load fine-tuned model first, fall back to base model
    try:
        model = tf.keras.models.load_model(f'{model_name}_subspecies_model_finetuned.keras')
        print("  Using fine-tuned model")
    except:
        model = tf.keras.models.load_model(f'{model_name}_subspecies_model.keras')
        print("  Using base model")

    # Evaluate
    test_results_metrics = model.evaluate(test_dataset, verbose=0)
    test_loss = test_results_metrics[0]
    test_acc = test_results_metrics[1]
    test_precision = test_results_metrics[2] if len(test_results_metrics) > 2 else 0
    test_recall = test_results_metrics[3] if len(test_results_metrics) > 3 else 0

    # Calculate mAP on test set
    print("  Calculating mAP...")
    mAP_test, ap_scores = calculate_map(model, test_dataset)

    test_results[model_name] = {
        'loss': test_loss,
        'accuracy': test_acc,
        'precision': test_precision,
        'recall': test_recall,
        'mAP': mAP_test
    }

    print(f"  ├─ Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")
    print(f"  ├─ Test Loss: {test_loss:.4f}")
    print(f"  ├─ Test Precision: {test_precision:.4f}")
    print(f"  ├─ Test Recall: {test_recall:.4f}")
    print(f"  └─ Test mAP: {mAP_test:.4f}")

# Save test results
with open('test_results.pkl', 'wb') as f:
    pickle.dump(test_results, f)



FINAL EVALUATION ON TEST SET

--- Evaluating ResNet50 on test set ---
  Using fine-tuned model
  Calculating mAP...
  Calculating mAP (this may take a few minutes)...
  ├─ Test Accuracy: 0.5714 (57.14%)
  ├─ Test Loss: 1.9603
  ├─ Test Precision: 0.0000
  ├─ Test Recall: 0.0000
  └─ Test mAP: 0.3654

--- Evaluating DenseNet121 on test set ---
  Using fine-tuned model
  Calculating mAP...
  Calculating mAP (this may take a few minutes)...
  ├─ Test Accuracy: 0.3238 (32.38%)
  ├─ Test Loss: 2.3005
  ├─ Test Precision: 0.0000
  ├─ Test Recall: 0.0000
  └─ Test mAP: 0.2059

--- Evaluating MobileNetV3 on test set ---
  Using fine-tuned model
  Calculating mAP...
  Calculating mAP (this may take a few minutes)...
  ├─ Test Accuracy: 0.6095 (60.95%)
  ├─ Test Loss: 2.0955
  ├─ Test Precision: 0.0000
  ├─ Test Recall: 0.0000
  └─ Test mAP: 0.3796


In [30]:
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)

comparison_data = []
for model_name in model_names:
    tr = test_results[model_name]
    tt = training_records[model_name]['training_time_minutes']
    comparison_data.append({
        'Model': model_name,
        'Test Accuracy': f"{tr['accuracy']:.4f}",
        'Test mAP': f"{tr['mAP']:.4f}",
        'Precision': f"{tr['precision']:.4f}",
        'Recall': f"{tr['recall']:.4f}",
        'Training Time (min)': f"{tt:.1f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n", comparison_df.to_string(index=False))

# Save comparison to CSV for Data Analyst
comparison_df.to_csv('model_comparison.csv', index=False)
print("\n✓ Comparison saved to 'model_comparison.csv'")


MODEL COMPARISON SUMMARY

       Model Test Accuracy Test mAP Precision Recall Training Time (min)
   ResNet50        0.5714   0.3654    0.0000 0.0000                 1.1
DenseNet121        0.3238   0.2059    0.0000 0.0000                 2.8
MobileNetV3        0.6095   0.3796    0.0000 0.0000                 1.1

✓ Comparison saved to 'model_comparison.csv'


In [31]:
model_parameters = {
    'ResNet50': 23.5,      # million parameters
    'DenseNet121': 7.0,    # million parameters
    'MobileNetV3': 5.4     # million parameters
}

print("\n" + "="*60)
print("MODEL PARAMETERS")
print("="*60)
for model_name, params in model_parameters.items():
    print(f"  {model_name}: {params} million parameters")


MODEL PARAMETERS
  ResNet50: 23.5 million parameters
  DenseNet121: 7.0 million parameters
  MobileNetV3: 5.4 million parameters
